# Evaluating Recommenders — Exercises

Companion to the note [Evaluating Recommenders](Evaluating%20Recommenders.md).

Practise the offline ranking metrics (Precision/Recall/HitRate@K, MRR, MAP@K, NDCG@K, AUC) by hand and from scratch, then the evaluation protocol questions that decide whether a number means anything: temporal vs random splits, sampled vs full ranking, beyond-accuracy metrics, and off-policy (IPS) estimation from logged data. Data are small synthetic ranked lists and interaction logs.

**18 exercises** · ✍️ By hand ×7 · 🧠 Concept ×4 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from scipy.stats import binom
from sklearn.metrics import ndcg_score, roc_auc_score
rng = np.random.default_rng(0)

## Part A · Metrics by hand

### Exercise 1 · Precision and recall at K
*✍️ By hand · ★☆☆*

A model ranks items `[A, B, C, D, E]` for a user whose relevant (held-out) items are $\{A, C, F\}$.
Compute Precision@3, Recall@3, Precision@5 and Recall@5. Why can Recall@5 never reach 1 here?

In [ ]:
p3 = None
r3 = None
p5 = None
r5 = None

_rank = list('ABCDE'); _rel = set('ACF')
for _k, _p, _r in [(3, p3, r3), (5, p5, r5)]:
    _h = sum(x in _rel for x in _rank[:_k])
    check(f'P@{_k}', _p, _h / _k); check(f'R@{_k}', _r, _h / len(_rel))

<details>
<summary><b>💡 Hint</b></summary>

Hits in the top 3: A and C. Hits in the top 5: still A and C.

</details>

<details>
<summary><b>✅ Solution</b></summary>

P@3 $= 2/3$, R@3 $= 2/3$, P@5 $= 2/5 = 0.4$, R@5 $= 2/3$.
F is not in the list at all, so recall is capped. Precision@K also depends on how many relevant items exist: with 1 relevant item, P@5 can be at most 0.2.
That is why results are usually reported as recall/NDCG under full ranking rather than raw precision.

```python
p3 = 2/3
r3 = 2/3
p5 = 0.4
r5 = 2/3
```

</details>

### Exercise 2 · HitRate and MRR
*✍️ By hand · ★☆☆*

Three users each have one held-out item. Its rank in their recommendation list is 1, 3 and 15 respectively.
Compute HitRate@10, HitRate@3 and MRR@10 (a user whose item is outside the top 10 contributes 0).

In [ ]:
hr10 = None
hr3 = None
mrr10 = None

_r = np.array([1, 3, 15])
check('HR@10', hr10, np.mean(_r <= 10)); check('HR@3', hr3, np.mean(_r <= 3))
check('MRR@10', mrr10, np.mean(np.where(_r <= 10, 1 / _r, 0)))

<details>
<summary><b>💡 Hint</b></summary>

MRR averages $1/\text{rank}$ of the first relevant item.

</details>

<details>
<summary><b>✅ Solution</b></summary>

HR@10 $= 2/3$, HR@3 $= 2/3$, MRR@10 $= (1 + 1/3 + 0)/3 = 4/9 \approx 0.444$.
MRR rewards putting the hit at the very top. A move from rank 1 to rank 2 halves that user's contribution, while HR@10 does not notice it.

```python
hr10 = 2/3
hr3 = 2/3
mrr10 = 4/9
```

</details>

### Exercise 3 · Average precision
*✍️ By hand · ★★☆*

A ranked list of 5 items has relevance pattern $(1, 0, 1, 0, 1)$ and the user has exactly 3 relevant items.
Compute AP@5 $= \frac{1}{\min(K, |\text{rel}|)}\sum_{k=1}^{K} P@k\cdot \mathrm{rel}_k$. What would AP@5 be for the pattern $(0,1,0,1,1)$?

In [ ]:
ap_a = None
ap_b = None

def _ap(rel, n_rel):
    rel = np.asarray(rel); prec = np.cumsum(rel) / np.arange(1, len(rel) + 1)
    return np.sum(prec * rel) / min(len(rel), n_rel)
check('AP pattern a', ap_a, _ap([1, 0, 1, 0, 1], 3)); check('AP pattern b', ap_b, _ap([0, 1, 0, 1, 1], 3))

<details>
<summary><b>💡 Hint</b></summary>

Pattern a: precision at the hits is $1/1$, $2/3$, $3/5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(1 + 2/3 + 3/5)/3 = (15+10+9)/45 = 34/45 \approx 0.756$.
(b) Hits at ranks 2, 4, 5: $(1/2 + 2/4 + 3/5)/3 = 1.6/3 \approx 0.533$.
Both lists contain the same 3 hits; AP rewards the one that ranks them higher. MAP@K is the mean of AP@K over users.

```python
ap_a = 34/45
ap_b = 1.6/3
```

</details>

### Exercise 4 · NDCG with graded relevance
*✍️ By hand · ★★☆*

A user's relevant items have grades 3, 2 and 2 (all others 0). The model's top-3 contains items with grades $(2, 3, 0)$.
With $DCG@K=\sum_{k=1}^K\frac{2^{rel_k}-1}{\log_2(k+1)}$, compute DCG@3, IDCG@3 and NDCG@3.

In [ ]:
dcg3 = None
idcg3 = None
ndcg3 = None

_d = lambda r: sum((2 ** x - 1) / np.log2(k + 2) for k, x in enumerate(r))
check('DCG@3', dcg3, _d([2, 3, 0]), tol=1e-3); check('IDCG@3', idcg3, _d([3, 2, 2]), tol=1e-3)
check('NDCG@3', ndcg3, ndcg_score([[7, 3, 3, 0]], [[3, 4, 1, 2]], k=3), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Gains: grade 3 → 7, grade 2 → 3. Discounts: $1$, $1/\log_2 3 \approx 0.631$, $1/2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The ideal top-3 is grades $(3, 2, 2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

DCG@3 $= 3/1 + 7/\log_2 3 + 0 = 3 + 4.417 = 7.417$.
IDCG@3 $= 7 + 3/\log_2 3 + 3/2 = 7 + 1.893 + 1.5 = 10.393$.
NDCG@3 $= 7.417/10.393 \approx 0.714$. Swapping the first two items would give 1.0, since the exponential gain heavily rewards putting the grade-3 item first.

```python
dcg3 = 3 + 7 / np.log2(3)
idcg3 = 7 + 3 / np.log2(3) + 1.5
ndcg3 = dcg3 / idcg3
```

</details>

### Exercise 5 · Per-user AUC
*✍️ By hand · ★☆☆*

For one user the model scores two positives at $0.9$ and $0.4$ and three negatives at $0.7$, $0.3$, $0.1$.
Compute the AUC as the fraction of (positive, negative) pairs ranked correctly. Why is AUC less suited than NDCG@10 for a feed showing 10 items?

In [ ]:
auc_user = None

check('AUC', auc_user, roc_auc_score([1, 1, 0, 0, 0], [.9, .4, .7, .3, .1]))

<details>
<summary><b>💡 Hint</b></summary>

There are $2\times 3 = 6$ pairs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$0.9$ beats all 3 negatives; $0.4$ beats $0.3$ and $0.1$ but not $0.7$. AUC $= 5/6 \approx 0.833$.
AUC weighs a mistake at rank 900 the same as at rank 1. Users only see the top of the list, so top-heavy metrics (NDCG@K, MRR) match the product better.
BPR optimises AUC ([[Collaborative Filtering and Matrix Factorization]]).

```python
auc_user = 5/6
```

</details>

### Exercise 6 · Why sampled metrics mislead
*✍️ By hand · ★★★*

A user's held-out item has **true rank 200** among $N=10\,000$ non-interacted items (199 items score higher), so the true HR@10 is 0.
With a *sampled* protocol, the item is ranked against only 100 negatives drawn uniformly at random.

1. What is the expected number of sampled negatives that score higher, and the expected sampled rank?
2. What is the probability that the item lands in the sampled top 10 (i.e. at most 9 sampled negatives above it)? Use a binomial model.

In [ ]:
exp_sampled_rank = None
p_hit_sampled = None

check('expected sampled rank', exp_sampled_rank, 1 + 100 * 199 / 10000)
check('P(hit@10 | sampled)', p_hit_sampled, binom.cdf(9, 100, 199 / 10000), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Each sampled negative scores higher with probability $p = 199/10000$, independently (approximately), so the count above is $\mathrm{Binomial}(100, p)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $100\cdot 0.0199 = 1.99$ negatives above on average, so the expected sampled rank is $\approx 2.99$.
2. $P(\mathrm{Bin}(100, 0.0199)\le 9) \approx 0.99997$: a near-certain "hit".

The sampled metric reports a hit for an item at true rank 200. It compresses the whole top of the ranking into "hit". Krichene & Rendle (2020) show this can even
**reverse** the order of two models. Prefer full ranking.

```python
exp_sampled_rank = 2.99
p_hit_sampled = binom.cdf(9, 100, 0.0199)
```

</details>

### Exercise 7 · Inverse propensity scoring by hand
*✍️ By hand · ★★☆*

A logging policy shows banner A with probability $0.8$ and B with $0.2$. Four logged impressions: (A, click), (A, no click), (B, click), (A, no click).
Estimate with IPS, $\hat V(\pi) = \frac1n\sum_t \frac{\pi(a_t)}{\pi_0(a_t)} r_t$, the click rate of the policies "always show B" and "always show A".
What is suspicious about the first estimate?

In [ ]:
v_B = None
v_A = None

_a = np.array(['A', 'A', 'B', 'A']); _r = np.array([1, 0, 1, 0]); _p0 = np.where(_a == 'A', .8, .2)
check('V(always B)', v_B, np.mean((_a == 'B') / _p0 * _r)); check('V(always A)', v_A, np.mean((_a == 'A') / _p0 * _r))

<details>
<summary><b>💡 Hint</b></summary>

For 'always B', $\pi(a_t)=1$ when $a_t = B$ and 0 otherwise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\hat V(B) = \frac14\cdot\frac{1}{0.2}\cdot1 = 1.25$; $\hat V(A) = \frac14\cdot\frac{1}{0.8}\cdot1 = 0.3125$.
A click-**rate** of 1.25 is impossible: IPS is unbiased but has **high variance** when the target policy relies on actions the logger rarely took (large weights $1/0.2$).
Fixes: self-normalised IPS (divide by the sum of weights instead of $n$), weight clipping, doubly-robust estimators, and logging with enough exploration.

```python
v_B = 1.25
v_A = 0.3125
```

</details>

## Part B · Protocol concepts

### Exercise 8 · Why random splits overestimate
*🧠 Concept · ★☆☆*

Explain two distinct ways a **random** train/test split of an interaction log leaks information that a deployed recommender would not have.
Why is leave-last-out better, and what leakage can it **still** contain?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about trends (an item that became popular next month) and about a user's own future.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Global future:** training data includes interactions from after the test interaction, e.g. the item's later popularity surge. The model "knows" what will trend.
2. **The user's own future:** the model sees what the user did after the test event (sequels, follow-up purchases), which makes predicting the test item trivial.

Leave-last-out removes (2): each user's most recent interaction is held out. But if users' timelines are not aligned, other users' interactions from **after** that time
are still in training (global leakage). A single **global time cutoff** (train before $t_0$, test after) is the most realistic. See [[Common Pitfalls]].

</details>

### Exercise 9 · Beyond accuracy
*🧠 Concept · ★☆☆*

Define coverage, diversity, novelty and serendipity in one line each, and give for each a product failure that accuracy metrics would not catch.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Popularity recommenders score well on accuracy. What else do they do badly?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Coverage:** share of the catalogue that is ever recommended. Failure: 95% of items never shown, and sellers leave.
- **Diversity:** dissimilarity within a list (e.g. 1 − intra-list similarity). Failure: ten nearly identical phone cases.
- **Novelty:** how unpopular/unknown recommended items are, e.g. mean $-\log_2 p(i)$. Failure: recommending only blockbusters the user already knows.
- **Serendipity:** relevant **and** unexpected (beyond what a simple baseline would show). Failure: correct but boring recommendations, and users stop exploring.

</details>

### Exercise 10 · Offline vs online
*🧠 Concept · ★★☆*

Offline NDCG went up 3%, but the A/B test shows no change in retention. Give three reasons this can happen.
What is **interleaving**, and why does it need far fewer users than an A/B test?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Offline data was logged by the old system; online metrics measure long-term behaviour.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Reasons: (1) **exposure bias**: logged data only contains reactions to what the old model showed, so offline metrics reward imitating it. (2) The offline metric (next click)
is not the business goal (retention, satisfaction). Optimising clicks can promote clickbait. (3) Gains concentrate on users or items that matter little, or are within noise.
Novelty effects and seasonality also affect A/B results.

**Interleaving:** both rankers' lists are merged into one list (e.g. team-draft), shown to the **same** user, and clicks are credited to the ranker that contributed the item.
Each user compares both models directly, which removes between-user variance, so it detects preferences with 10–100× fewer users than A/B tests. It does not measure absolute business metrics, though.

</details>

### Exercise 11 · Choosing the metric
*🧠 Concept · ★☆☆*

Which offline metric fits best, and why? (a) Rating prediction for a star display. (b) A "next song" autoplay with one correct answer. (c) A search-like results page
with graded relevance labels. (d) A retrieval stage returning 500 candidates for a ranker.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Is there one or many relevant items? Graded or binary? Does position within the list matter?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **RMSE/MAE**: the predicted value itself is shown.
(b) **MRR** or **HitRate@K**: there is a single target, and its position matters (MRR).
(c) **NDCG@K**: graded relevance with a position discount.
(d) **Recall@500**: the job is not to lose relevant items. Order within the 500 is the ranker's job.

</details>

## Part C · Code from scratch

### Exercise 12 · Top-K metrics in NumPy
*💻 Code · ★☆☆*

Implement `topk_metrics(ranked, relevant, K)` returning a dict with `precision`, `recall`, `hit`, `rr` (reciprocal rank of the first relevant item within the top K, else 0)
for one user. Then average over the toy users below.

In [ ]:
toy_ranked = [[3, 1, 7, 2, 9], [5, 4, 3, 2, 1], [8, 6, 0, 1, 2]]
toy_relevant = [{1, 2}, {7}, {0, 8, 9}]

def topk_metrics(ranked, relevant, K):
    return None

avg_metrics = None   # dict of averages over the 3 users at K=3

_ref = {'precision': [], 'recall': [], 'hit': [], 'rr': []}
for _rk, _rl in zip(toy_ranked, toy_relevant):
    _top = _rk[:3]; _h = [x in _rl for x in _top]
    _ref['precision'].append(sum(_h) / 3); _ref['recall'].append(sum(_h) / len(_rl)); _ref['hit'].append(float(any(_h)))
    _ref['rr'].append(next((1 / (k + 1) for k, x in enumerate(_h) if x), 0.0))
if avg_metrics is not None:
    for _m in _ref:
        check(_m, avg_metrics[_m], np.mean(_ref[_m]))
else:
    check('metrics', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`hits = [i in relevant for i in ranked[:K]]`; `rr = 1/(first index of a hit + 1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At K=3: precision $= (1/3 + 0 + 2/3)/3 = 1/3$, recall $= (1/2 + 0 + 2/3)/3 \approx 0.389$, hit $= 2/3$, MRR $= (1/2 + 0 + 1)/3 = 0.5$.
Always average **per user** first ("macro"), so heavy users do not dominate.

```python
toy_ranked = [[3, 1, 7, 2, 9], [5, 4, 3, 2, 1], [8, 6, 0, 1, 2]]
toy_relevant = [{1, 2}, {7}, {0, 8, 9}]

def topk_metrics(ranked, relevant, K):
    hits = [i in relevant for i in ranked[:K]]
    first = next((k for k, h in enumerate(hits) if h), None)
    return {'precision': sum(hits) / K, 'recall': sum(hits) / len(relevant),
            'hit': float(any(hits)), 'rr': 0.0 if first is None else 1 / (first + 1)}

_all = [topk_metrics(r, s, 3) for r, s in zip(toy_ranked, toy_relevant)]
avg_metrics = {m: float(np.mean([d[m] for d in _all])) for m in _all[0]}
print(avg_metrics)
```

</details>

### Exercise 13 · NDCG@K, checked against sklearn
*💻 Code · ★★☆*

Implement `ndcg_at_k(scores, grades, K)` for one user: `scores` are model scores for all items, `grades` the true relevance grades. Use the note's exponential gain $2^{rel}-1$
and ideal ordering from `grades`. sklearn's `ndcg_score` uses **linear** gain, so to compare, pass it `2**grades - 1` as `y_true`.
Evaluate on the random example below.

In [ ]:
_g = np.random.default_rng(1)
grades_ex = _g.integers(0, 4, size=30)
scores_ex = grades_ex + _g.normal(0, 1.5, size=30)

def ndcg_at_k(scores, grades, K):
    return None

ndcg10 = ndcg_at_k(scores_ex, grades_ex, 10)

check('NDCG@10', ndcg10, ndcg_score([2 ** grades_ex - 1], [scores_ex], k=10))

<details>
<summary><b>💡 Hint</b></summary>

`order = np.argsort(-scores)[:K]`; `disc = 1/np.log2(np.arange(2, K+2))`; IDCG uses `np.sort(grades)[::-1][:K]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With linear gains, grade 3 is worth 3× grade 1. With exponential gains it is worth 7×, so the metric is much more top-grade-focused. Always state which variant you report.
Papers differ, and numbers are not comparable across the two.

```python
_g = np.random.default_rng(1)
grades_ex = _g.integers(0, 4, size=30)
scores_ex = grades_ex + _g.normal(0, 1.5, size=30)

def ndcg_at_k(scores, grades, K):
    disc = 1 / np.log2(np.arange(2, K + 2))
    gains = 2.0 ** np.asarray(grades) - 1
    dcg = np.sum(gains[np.argsort(-scores)[:K]] * disc)
    idcg = np.sum(np.sort(gains)[::-1][:K] * disc)
    return dcg / idcg if idcg > 0 else 0.0

ndcg10 = ndcg_at_k(scores_ex, grades_ex, 10)
print(round(ndcg10, 4))
```

</details>

### Exercise 14 · AUC by counting pairs
*💻 Code · ★☆☆*

Implement `pairwise_auc(scores, labels)` by comparing every positive with every negative (ties count ½). Then implement a faster $O(n\log n)$ version `rank_auc`
using ranks (Mann–Whitney U: $\mathrm{AUC} = \frac{\sum_{i\in +}\mathrm{rank}_i - n_+(n_++1)/2}{n_+n_-}$, average ranks for ties).

In [ ]:
from scipy.stats import rankdata
_g = np.random.default_rng(2)
lab_ex = (_g.random(200) < 0.3).astype(int)
sc_ex = np.round(lab_ex + _g.normal(0, 1, 200), 1)   # rounded -> ties exist

def pairwise_auc(scores, labels):
    return None

def rank_auc(scores, labels):
    return None

auc_pair = pairwise_auc(sc_ex, lab_ex)
auc_rank = rank_auc(sc_ex, lab_ex)

check('pairwise AUC', auc_pair, roc_auc_score(lab_ex, sc_ex))
check('rank AUC', auc_rank, roc_auc_score(lab_ex, sc_ex))

<details>
<summary><b>💡 Hint</b></summary>

Pairwise: `d = pos[:, None] - neg[None, :]`; `np.mean((d > 0) + 0.5 * (d == 0))`. Ranks: `rankdata(scores)` averages ties.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both equal sklearn's `roc_auc_score`. The rank version is what you use at scale. The pairwise form shows why AUC is the probability that a random positive outranks a random negative,
the quantity BPR's loss approximates.

```python
from scipy.stats import rankdata
_g = np.random.default_rng(2)
lab_ex = (_g.random(200) < 0.3).astype(int)
sc_ex = np.round(lab_ex + _g.normal(0, 1, 200), 1)

def pairwise_auc(scores, labels):
    pos, neg = scores[labels == 1], scores[labels == 0]
    d = pos[:, None] - neg[None, :]
    return float(np.mean((d > 0) + 0.5 * (d == 0)))

def rank_auc(scores, labels):
    r = rankdata(scores)
    n_pos = labels.sum(); n_neg = len(labels) - n_pos
    return float((r[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

auc_pair = pairwise_auc(sc_ex, lab_ex)
auc_rank = rank_auc(sc_ex, lab_ex)
```

</details>

### Exercise 15 · Leave-last-out split
*💻 Code · ★★☆*

Given an interaction log (`user, item, ts`), implement `leave_last_out(log)` returning `(train, test)`, where `test` holds each user's **most recent** interaction
(only for users with ≥2 interactions) and `train` everything else.

In [ ]:
_g = np.random.default_rng(3)
log_small = pd.DataFrame({'user': _g.integers(0, 30, 300), 'item': _g.integers(0, 50, 300), 'ts': _g.permutation(300)})

def leave_last_out(log):
    return None

split = leave_last_out(log_small)

if split is not None:
    _tr, _te = split
    _cnt = log_small.user.value_counts()
    check('one test row per user with >= 2 interactions', sorted(_te.user.tolist()), sorted(_cnt[_cnt >= 2].index.tolist()))
    check('test rows are the latest per user', bool((_te.set_index('user').ts == log_small.groupby('user').ts.max().loc[_te.user].values).all()), True)
    check('train + test = log, no overlap', len(_tr) + len(_te) == len(log_small) and len(set(_tr.index) & set(_te.index)) == 0, True)
    check('no test interaction precedes that user\'s training data', bool((_tr.groupby('user').ts.max().loc[_te.user].values < _te.ts.values).all()), True)
else:
    check('split', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`idx = log.groupby('user').ts.idxmax()`; filter to users with `size() >= 2`; `test = log.loc[idx]`, `train = log.drop(idx)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`groupby().idxmax()` gives the index of the latest event per user. Users with a single interaction stay in training only, since there is nothing left to learn from if their one event is held out.

```python
_g = np.random.default_rng(3)
log_small = pd.DataFrame({'user': _g.integers(0, 30, 300), 'item': _g.integers(0, 50, 300), 'ts': _g.permutation(300)})

def leave_last_out(log):
    sizes = log.groupby('user').size()
    eligible = log[log.user.isin(sizes[sizes >= 2].index)]
    idx = eligible.groupby('user').ts.idxmax()
    return log.drop(idx), log.loc[idx]

split = leave_last_out(log_small)
```

</details>

### Exercise 16 · Random vs temporal split: measuring the leak
*💻 Code · ★★★*

Simulate a log with **trends**: 300 users, 100 items, each user has 8 interactions at random times in $[0, 200)$; at time $t$ the item is drawn with probability
$\propto \exp(-((i - c_t)/8)^2)$, where $c_t = t/2$ is a "trending" centre moving through the catalogue (code below).

Evaluate the **most-popular** recommender (popularity counted on train, excluding the user's train items) by HR@10 under (a) leave-last-out and (b) leave-one-**random**-out
(`groupby('user').sample(1, random_state=0)`). Store `hr_temporal` and `hr_random`. Explain the gap.

In [ ]:
_g = np.random.default_rng(0)
_rows = []
for _u in range(300):
    for _t in np.sort(_g.choice(200, 8, replace=False)):
        _w = np.exp(-((np.arange(100) - _t / 2) / 8) ** 2)
        _rows.append((_u, _g.choice(100, p=_w / _w.sum()), _t))
log_trend = pd.DataFrame(_rows, columns=['user', 'item', 'ts'])

def hr_popular(train, test, K=10):
    return None

hr_temporal = None
hr_random = None

def _hr(train, test, K=10):
    pop = train.item.value_counts()
    hits = []
    for u, it in zip(test.user, test.item):
        seen = set(train.item[train.user == u])
        hits.append(it in [i for i in pop.index if i not in seen][:K])
    return float(np.mean(hits))
_i1 = log_trend.groupby('user').ts.idxmax()
_i2 = log_trend.groupby('user').sample(1, random_state=0).index
if hr_temporal is not None:
    check('HR@10 temporal', hr_temporal, _hr(log_trend.drop(_i1), log_trend.loc[_i1]))
    check('HR@10 random', hr_random, _hr(log_trend.drop(_i2), log_trend.loc[_i2]))
    check('random split looks much better', hr_random > 5 * hr_temporal, True)
else:
    check('split comparison', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`pop = train.item.value_counts()` is sorted by count; for each test row walk down `pop.index`, skipping the user's train items, and take the first 10.

</details>

<details>
<summary><b>✅ Solution</b></summary>

HR@10 ≈ 0.10 with the random split vs ≈ 0.003 with leave-last-out. Under a random split, the held-out interaction usually happened in the middle of the timeline, when the items
that are globally popular in the training data were trending. Under leave-last-out, users' last events happen later than most of the training data, so global popularity
points to items that are no longer trending. The random split rewards a model for "knowing the past and future". The same model's quality differs by 30× depending only on the protocol.

```python
_g = np.random.default_rng(0)
_rows = []
for _u in range(300):
    for _t in np.sort(_g.choice(200, 8, replace=False)):
        _w = np.exp(-((np.arange(100) - _t / 2) / 8) ** 2)
        _rows.append((_u, _g.choice(100, p=_w / _w.sum()), _t))
log_trend = pd.DataFrame(_rows, columns=['user', 'item', 'ts'])

def hr_popular(train, test, K=10):
    pop = train.item.value_counts().index.to_numpy()
    seen = train.groupby('user').item.apply(set)
    hits = []
    for u, it in zip(test.user, test.item):
        s = seen.get(u, set())
        recs = [i for i in pop if i not in s][:K]
        hits.append(it in recs)
    return float(np.mean(hits))

_i1 = log_trend.groupby('user').ts.idxmax()
_i2 = log_trend.groupby('user').sample(1, random_state=0).index
hr_temporal = hr_popular(log_trend.drop(_i1), log_trend.loc[_i1])
hr_random = hr_popular(log_trend.drop(_i2), log_trend.loc[_i2])
print('temporal', round(hr_temporal, 4), '| random', round(hr_random, 4))
```

</details>

### Exercise 17 · Sampled vs full-ranking HitRate
*💻 Code · ★★☆*

For 500 users, a mediocre model gives the held-out item a score drawn from $\mathcal N(1, 1)$ and each of 2000 negatives a score from $\mathcal N(0,1)$ (code below).
Implement `hr_full(pos, neg, K)` (rank against **all** negatives) and `hr_sampled(pos, neg, K, n_samp, seed)` (rank against `n_samp` negatives sampled per user without replacement).
Compare HR@10 for both.

In [ ]:
_g = np.random.default_rng(4)
pos_s = _g.normal(1, 1, 500)
neg_s = _g.normal(0, 1, (500, 2000))

def hr_full(pos, neg, K=10):
    return None

def hr_sampled(pos, neg, K=10, n_samp=100, seed=0):
    return None

hr_f = hr_full(pos_s, neg_s)
hr_s = hr_sampled(pos_s, neg_s)

check('full HR@10', hr_f, np.mean((neg_s > pos_s[:, None]).sum(1) < 10))
if hr_s is not None:
    check('sampled HR@10 is far higher', hr_s > 3 * hr_f, True)

<details>
<summary><b>💡 Hint</b></summary>

Hit@K ⇔ fewer than K negatives score above the positive: `(neg > pos[:, None]).sum(1) < K`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Full HR@10 is ≈ 0.06, sampled HR@10 ≈ 0.37. The same model scores about 6× higher under sampling. Sampled numbers are not comparable to full-ranking numbers, and the
ranking of different models can change under sampling. Report full-ranking metrics whenever the catalogue size allows.

```python
_g = np.random.default_rng(4)
pos_s = _g.normal(1, 1, 500)
neg_s = _g.normal(0, 1, (500, 2000))

def hr_full(pos, neg, K=10):
    return float(np.mean((neg > pos[:, None]).sum(1) < K))

def hr_sampled(pos, neg, K=10, n_samp=100, seed=0):
    g = np.random.default_rng(seed)
    hits = [(g.choice(neg[u], n_samp, replace=False) > pos[u]).sum() < K for u in range(len(pos))]
    return float(np.mean(hits))

hr_f = hr_full(pos_s, neg_s)
hr_s = hr_sampled(pos_s, neg_s)
print('full', hr_f, '| sampled', hr_s)
```

</details>

### Exercise 18 · Off-policy evaluation with IPS
*💻 Code · ★★★*

Five banners have true click rates `ctr = [0.10, 0.30, 0.20, 0.05, 0.15]`. A logging policy $\pi_0 = \mathrm{softmax}(2, 0, 0, 0, 0)$ (mostly banner 0) generated $n=20\,000$ impressions.
We want the click rate of a new policy $\pi_1 = (0, 0.5, 0.5, 0, 0)$ **without** deploying it.

1. Compute the true value $V(\pi_1) = \sum_a \pi_1(a)\,\mathrm{ctr}_a$.
2. Implement `ips(actions, rewards, p0, p1)` and `snips` (self-normalised: divide by $\sum_t w_t$ instead of $n$).
3. Compare with the **naive** estimate: the average click rate of the logs.

In [ ]:
ctr_b = np.array([0.10, 0.30, 0.20, 0.05, 0.15])
p0 = np.exp([2, 0, 0, 0, 0]) / np.exp([2, 0, 0, 0, 0]).sum()
p1 = np.array([0, .5, .5, 0, 0])
_g = np.random.default_rng(5)
acts = _g.choice(5, 20000, p=p0)
rews = (_g.random(20000) < ctr_b[acts]).astype(float)

v_true = None
def ips(actions, rewards, p0, p1):
    return None
def snips(actions, rewards, p0, p1):
    return None
v_ips = ips(acts, rews, p0, p1)
v_snips = snips(acts, rews, p0, p1)
v_naive = None

check('true value', v_true, p1 @ ctr_b)
if v_ips is not None:
    _w = p1[acts] / p0[acts]
    check('IPS', v_ips, np.mean(_w * rews)); check('SNIPS', v_snips, np.sum(_w * rews) / np.sum(_w))
    check('IPS is close to the truth', abs(v_ips - p1 @ ctr_b) < 0.03, True)
    check('naive estimate is far off', abs(v_naive - p1 @ ctr_b) > 0.08, True)

<details>
<summary><b>💡 Hint</b></summary>

Weights $w_t = \pi_1(a_t)/\pi_0(a_t)$: `p1[actions] / p0[actions]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$V(\pi_1) = 0.5\cdot0.3+0.5\cdot0.2 = 0.25$. IPS and SNIPS land close to 0.25, while the naive log CTR (≈0.13) mainly reflects banner 0, which $\pi_1$ never shows.
IPS needs $\pi_0(a) > 0$ wherever $\pi_1(a) > 0$ (**support**), which is why production systems keep some exploration. The weights here are up to $0.5/0.088\approx5.7$. With
rarer logged actions the variance explodes (Exercise 7).

```python
ctr_b = np.array([0.10, 0.30, 0.20, 0.05, 0.15])
p0 = np.exp([2, 0, 0, 0, 0]) / np.exp([2, 0, 0, 0, 0]).sum()
p1 = np.array([0, .5, .5, 0, 0])
_g = np.random.default_rng(5)
acts = _g.choice(5, 20000, p=p0)
rews = (_g.random(20000) < ctr_b[acts]).astype(float)

v_true = float(p1 @ ctr_b)
def ips(actions, rewards, p0, p1):
    return float(np.mean(p1[actions] / p0[actions] * rewards))
def snips(actions, rewards, p0, p1):
    w = p1[actions] / p0[actions]
    return float(np.sum(w * rewards) / np.sum(w))
v_ips = ips(acts, rews, p0, p1)
v_snips = snips(acts, rews, p0, p1)
v_naive = float(rews.mean())
print('true', v_true, '| IPS', round(v_ips, 4), '| SNIPS', round(v_snips, 4), '| naive', round(v_naive, 4))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Evaluating Recommenders](Evaluating%20Recommenders.md).*